## 1. Import & Setup

In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [ ]:
load_dotenv()

## 2. Load Dataset - Medication on Admission

In [ ]:
df = pd.read_parquet(f'{os.environ["DATA_PATH"]}/datasets/medication_on_admission.parquet')
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")


In [ ]:
df.head()

In [ ]:
df.info()

## 3. Data Cleaning

- Check and update columns data types;
- Drop NA values for meds_on_admission column;

In [ ]:
df1 = df.copy()

In [ ]:
df1.info()

In [ ]:
data_types_cols = ['note_id', 'subject_id', 'hadm_id', 'patient_gender', 'meds_on_admission', 'text']

df1[data_types_cols] = df1[data_types_cols].astype("string")

In [ ]:
df1['patient_age_group_on_admission'] = df1['patient_age_group_on_admission'].astype("category")

**To ensure we obtain a representative sample from the 'medication on admission' dataset, we need to remove rows with NULL or NA values in the following key columns:**
- meds_on_admission
- patient_gender
- patient_age_group_on_admission

In [ ]:
# Only drop rows where a specific column is NaN
df2 = df1.dropna(subset=['meds_on_admission', 'patient_gender', 'patient_age_group_on_admission'])

In [ ]:
df2.info()

## 4. Univariate analysis

Don't trust the mean alone. Look at the full shape: skew, tails,
zero-inflation, and values that suggest a log transform later.

### 4.1 Univariate Analysis - Numeric Variables

In [ ]:
df2.describe()

In [ ]:
num_cols = ['meds_on_admission_length']

if num_cols:
    desc = df2[num_cols].describe().T
    desc["skew"] = df2[num_cols].skew()
    desc["kurtosis"] = df2[num_cols].kurtosis()
    desc["n_zeros"] = (df2[num_cols] == 0).sum()
    display(desc)


In [ ]:
for c in num_cols:
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
    df2[c].dropna().hist(bins=40, ax=ax[0]); ax[0].set_title(f"{c} — Histogram")
    sns.boxplot(x=df2[c], ax=ax[1]); ax[1].set_title(f"{c} — Boxplot")
    plt.tight_layout(); plt.show()


In [ ]:
df2[df2['meds_on_admission_length']==26568.0]

**Note:**
- As you can see in the value above, the note_id `14266723-DS-19` has lots of whitspaces.

#### 4.1.1 Remove whitspace from meds_on_admission and calculate the meds_on_admission_length again

- Remove whitspace like spaces, tabs and newlines from meds_on_admission column.

In [ ]:
df3 = df2.copy()

df3.drop(columns='meds_on_admission_length')

In [ ]:
def clean_note(text):
    """Clean one note and return it as a tidy string."""
    # If the value isn't text (e.g. NaN), return an empty string.
    if not isinstance(text, str):
        return ""
 
    # Make all line endings the same.
    text = text.replace("\r\n", "\n").replace("\r", "\n")
 
    # Trim spaces at the start/end of every line.
    lines = [line.strip() for line in text.split("\n")]
    text = "\n".join(lines)
 
    # Turn runs of spaces or tabs into a single space.
    text = re.sub(r"[ \t]+", " ", text)
 
    # Reduce 3+ blank lines to a single blank line (keeps paragraph breaks).
    text = re.sub(r"\n{3,}", "\n\n", text)
 
    # Trim the whole thing.
    return text.strip()
 
 
def clean_column(df, column):
    """Add a cleaned version of a text column as '<column>_clean'."""
    df = df.copy()
    df[column + "_cleaned"] = df[column].apply(clean_note)
    return df

In [ ]:
df3 = clean_column(df3, 'meds_on_admission')

In [ ]:
df3.info()

In [ ]:
# Add column meds_on_admission_length
df3['meds_on_admission_length'] = df3['meds_on_admission_cleaned'].str.len()

In [ ]:
df2.describe()

In [ ]:
df3.describe()

In [ ]:
df3.info()

In [ ]:
df4 = df3.drop(columns=['meds_on_admission'])

### 4.2 Univariate Analysis of Categorical Variables

-------